# TCM-Classics-Bench · 模型测评 Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/TCM-Classics-Bench/blob/claude/tcm-classics-bench-dataset-tu71dt/notebooks/TCM_Classics_Bench_Eval.ipynb)

在中医古籍测评基准上**评测任意大模型**，并统计性能分数。

- **被测模型**支持四家 provider：Anthropic / Azure / **Poe** / **LiteLLM**（litellm 可路由 100+ 模型）。
- **Prompt 模式**：`zero_shot` / `few_shot` / `cot`（思维链）。
- **自动评分**：单选题→准确率(accuracy)；NER→F1；方剂解析→F1；句读→边界 F1。
- **实时进度条 + 预计剩余时间(ETA)**，结果**逐题实时写入 Google Drive**（断线不丢、可断点续跑）。
- 最终输出**每任务分数 + 总分**。

> 默认评测仓库内置的 **NER 子集 + T1/T6** 题（答案确定、可自动评分）。
> 你也可以指向第一份 Notebook 用 LLM 生成、存到 Drive 的难题（含单选题）。
> *运行环境：代码执行程序 → 全部运行；无需 GPU。*

## 1. 环境准备 · Clone & install

In [ ]:
import os
REPO = "https://github.com/pariskang/TCM-Classics-Bench.git"
BRANCH = "claude/tcm-classics-bench-dataset-tu71dt"   # 合并到 main 后改成 "main"
if not os.path.exists("/content/TCM-Classics-Bench"):
    !git clone --branch $BRANCH --depth 1 $REPO /content/TCM-Classics-Bench
%cd /content/TCM-Classics-Bench
!pip install -q tqdm pandas
print("ready")

## 2. 选择被测模型 / Prompt 模式，并填写密钥

In [ ]:
#@title provider / 模型 / prompt 模式 + 密钥 { display-mode: "form" }
import getpass, os
PROVIDER = "poe"        #@param ["anthropic", "azure", "poe", "litellm"]
MODEL = "Claude-Sonnet-4"  #@param {type:"string"}
MODE = "zero_shot"      #@param ["zero_shot", "few_shot", "cot"]
SHOTS = 3               #@param {type:"integer"}
MODEL = MODEL.strip() or None

def setkey(name):
    if not os.environ.get(name):
        os.environ[name] = getpass.getpass(f"{name}: ")

if PROVIDER == "anthropic":
    !pip install -q anthropic
    setkey("ANTHROPIC_API_KEY")
elif PROVIDER == "poe":
    !pip install -q openai
    setkey("POE_API_KEY")
elif PROVIDER == "azure":
    !pip install -q openai
    for k in ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT"]:
        setkey(k)
    os.environ.setdefault("AZURE_OPENAI_API_VERSION", "2024-06-01")
elif PROVIDER == "litellm":
    !pip install -q litellm

from tcm_bench.llm import make_client
EVAL_CLIENT = make_client(PROVIDER, MODEL)
print("被测模型就绪 ->", PROVIDER, "| model:", EVAL_CLIENT.model, "| 模式:", MODE)

## 3. 挂载 Google Drive（实时存储评测结果）

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/tcm_classics_bench_eval"
os.makedirs(OUT_DIR, exist_ok=True)
print("结果目录:", OUT_DIR)

## 4. 组装测评集

默认：仓库内置 **NER 子集**（`data/bench_ner/`）+ **T1/T6**（`data/bench/`），全部可自动评分。
也可把 `DATASET` 指向你自己的 JSONL（例如第一份 Notebook 生成、存到 Drive 的难题集，含单选题）。

In [ ]:
#@title 选择测评集与规模 { display-mode: "form" }
DATASET = "repo"   #@param {type:"string"}
MAX_ITEMS = 300    #@param {type:"integer"}

import glob, json, random
from tcm_bench.evaluate import scorable

if DATASET == "repo":
    files = sorted(glob.glob("data/bench_ner/*.jsonl")) + sorted(glob.glob("data/bench/*.jsonl"))
else:
    files = [DATASET]   # 你的 JSONL 路径（如 /content/drive/MyDrive/.../tcm_llm_questions.jsonl）

items = []
for f in files:
    items += [json.loads(l) for l in open(f, encoding="utf-8") if l.strip()]
items = [it for it in items if scorable(it)]   # 只保留可自动评分的题
random.Random(7).shuffle(items)
if MAX_ITEMS and MAX_ITEMS > 0:
    items = items[:MAX_ITEMS]

import collections
print("可评分题目:", len(items))
print("按任务:", dict(collections.Counter(it["task_code"] for it in items)))

## 5. 评测（实时进度 + ETA + 实时写入 Drive + 断点续跑）

`tqdm` 会自动显示**已完成/总数、速度与预计剩余时间(ETA)**；每题评分结果**实时 flush 到 Drive**。
重复运行本 cell 会**接着上次继续**（按 `(item_id, mode)` 跳过已评的）。

In [ ]:
import os, json
from tqdm.auto import tqdm
from tcm_bench.evaluate import evaluate_dataset, aggregate

out_path = os.path.join(OUT_DIR, f"eval_{PROVIDER}_{MODE}.jsonl")

# 断点续跑：读取已评结果
records, done = [], set()
if os.path.exists(out_path):
    for line in open(out_path, encoding="utf-8"):
        line = line.strip()
        if not line:
            continue
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        records.append(r); done.add((r["item_id"], r["prompt_mode"]))
todo = [it for it in items if (it["item_id"], MODE) not in done]
print(f"待评 {len(todo)} 题（已完成 {len(done)}）；模式={MODE} 模型={EVAL_CLIENT.model}")

stats = {}
with open(out_path, "a", encoding="utf-8") as fh, \
     tqdm(total=len(todo), unit="题", desc=f"eval/{MODE}") as bar:   # tqdm 自带 ETA
    for rec in evaluate_dataset(todo, EVAL_CLIENT, mode=MODE, shots_pool=items,
                                n_shots=SHOTS, max_workers=16, stats=stats):
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); fh.flush()   # 实时落盘
        records.append(rec); bar.update(1)

print("调用统计:", stats)

## 6. 性能分数 · Scores

In [ ]:
import pandas as pd, json
from tcm_bench.evaluate import aggregate

summary = aggregate(records)
print(json.dumps(summary, ensure_ascii=False, indent=2))

df = pd.DataFrame(summary["per_task"]).T
df.index.name = "task"
print("\n每任务分数：")
display(df)
print(f"总分 (overall macro): {summary['overall_macro']}  |  "
      f"已评 {summary['n_scored']}  错误 {summary['n_errors']}")
# 存一份分数 JSON 到 Drive
import os
with open(os.path.join(OUT_DIR, f"scores_{PROVIDER}_{MODE}.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

## 7. 看几条评测明细（预测 vs 评分）

In [ ]:
import json
for r in records[:6]:
    print("="*70)
    print(f"[{r['task_code']}/{r['family']}] score={r.get('score')} correct={r.get('correct')}")
    if r.get("detail"): print("  detail:", json.dumps(r["detail"], ensure_ascii=False))
    print("  模型预测:", str(r.get("pred",""))[:200])

## 8.（可选）对比 Prompt 模式：zero-shot vs few-shot vs CoT

在一个小子集上分别用三种模式评测，比较总分。注意：这会额外消耗 API。

In [ ]:
#@title 三种 prompt 模式对比 { display-mode: "form" }
N_COMPARE = 60   #@param {type:"integer"}
import os, json, pandas as pd
from tqdm.auto import tqdm
from tcm_bench.evaluate import evaluate_dataset, aggregate

subset = items[:N_COMPARE]
rows = {}
for m in ["zero_shot", "few_shot", "cot"]:
    recs = []
    for rec in tqdm(evaluate_dataset(subset, EVAL_CLIENT, mode=m, shots_pool=items,
                                     n_shots=SHOTS, max_workers=16),
                    total=len(subset), desc=m):
        recs.append(rec)
    agg = aggregate(recs)
    row = {tc: v["score"] for tc, v in agg["per_task"].items()}
    row["overall"] = agg["overall_macro"]
    rows[m] = row
cmp = pd.DataFrame(rows).T
cmp.index.name = "prompt_mode"
print("不同 prompt 模式分数对比：")
display(cmp)

## 9. 下载结果

In [ ]:
from google.colab import files
import os
files.download(os.path.join(OUT_DIR, f"eval_{PROVIDER}_{MODE}.jsonl"))

---
仓库：**pariskang/TCM-Classics-Bench**（分支 `claude/tcm-classics-bench-dataset-tu71dt`）·
评测模块 `tcm_bench/evaluate.py` · 命令行 `python -m tcm_bench evaluate --help`。
题目仅作古籍理解测评，**不构成任何用药建议**。